In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# setting display options so dataframes are readable
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid')
print("imports done")

In [ ]:
import os

#auto-detect-enviornment - works in both Colab and VS Code
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = '/content/drive/MyDrive/ML/stock-lens/data/'
    print("running in Colab")
except ImportError:
    data_path = 'data/'
    print("running locally")

os.makedirs(data_path, exist_ok=True)
print("data path:", data_path)

In [ ]:
kaggle_files = ['prices-split-adjusted.csv', 'fundamentals.csv', 'securities.csv', 'prices.csv']
files_exist = all(os.path.exists(data_path + f) for f in kaggle_files)

if not files_exist:
    os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'
    os.environ['KAGGLE_KEY']      = 'your_api_key'
    os.system('pip install kaggle -q')
    os.system(f'kaggle datasets download -d dgawlik/nyse -p {data_path} --unzip')
    print("download complete")
else:
    print("files already present — skipping download")

# loading all three datasets
prices       = pd.read_csv(data_path + 'prices-split-adjusted.csv', parse_dates=['date'])
fundamentals = pd.read_csv(data_path + 'fundamentals.csv')
securities   = pd.read_csv(data_path + 'securities.csv')

print("\nprices shape:      ", prices.shape)
print("fundamentals shape:", fundamentals.shape)
print("securities shape:  ", securities.shape)
print("\ndate range:", prices['date'].min(), "to", prices['date'].max())
print("unique tickers:", prices['symbol'].nunique())

In [ ]:
# loading the annual financial metrics from SEC 10-K filings
fundamentals = pd.read_csv(data_path + 'fundamentals.csv')

print("shape:", fundamentals.shape)
print("\ncolumns:", fundamentals.columns.tolist())
fundamentals.head()

In [ ]:
# loading sector and industry info per ticker
securities = pd.read_csv(data_path + 'securities.csv')

print("shape:", securities.shape)
print("\ncolumns:", securities.columns.tolist())
securities.head()

In [ ]:
# checking how many nulls exist in each column
null_counts = prices.isnull().sum()
print("null counts in prices:\n", null_counts)

In [ ]:
print("null counts in fundamentals:\n", fundamentals.isnull().sum())
print("\nnull counts in securities:\n", securities.isnull().sum())

In [ ]:
# checking how many trading days each ticker has — should be roughly equal
days_per_ticker = prices.groupby('symbol')['date'].count().sort_values()

plt.figure(figsize=(12, 4))
plt.plot(days_per_ticker.values)
plt.title('trading days per ticker')
plt.xlabel('ticker rank')
plt.ylabel('number of trading days')
plt.tight_layout()
plt.show()

print("min days:", days_per_ticker.min(), "| max days:", days_per_ticker.max())
print("tickers with fewer than 1000 days:", (days_per_ticker < 1000).sum())

In [ ]:
# looking at the overall distribution of closing prices across all tickers
plt.figure(figsize=(10, 4))
sns.histplot(prices['close'], bins=100, kde=True)
plt.title('distribution of closing prices (all tickers, all dates)')
plt.xlabel('closing price ($)')
plt.tight_layout()
plt.show()

In [ ]:
# plotting closing price over time for 4 well-known tickers to sanity check the data
example_tickers = ['AAPL', 'GOOGL', 'JPM', 'XOM']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, ticker in enumerate(example_tickers):
    ticker_data = prices[prices['symbol'] == ticker]
    axes[i].plot(ticker_data['date'], ticker_data['close'])
    axes[i].set_title(ticker)
    axes[i].set_xlabel('date')
    axes[i].set_ylabel('close price ($)')

plt.suptitle('closing price over time — sample tickers', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# checking how many tickers fall into each sector
sector_counts = securities['GICS Sector'].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=sector_counts.values, y=sector_counts.index, palette='Blues_r')
plt.title('number of tickers per sector')
plt.xlabel('count')
plt.tight_layout()
plt.show()

print(sector_counts)

In [ ]:
# checking total market volume over time to spot any anomalies
daily_volume = prices.groupby('date')['volume'].sum()

plt.figure(figsize=(12, 4))
plt.plot(daily_volume.index, daily_volume.values)
plt.title('total market volume over time')
plt.xlabel('date')
plt.ylabel('total volume')
plt.tight_layout()
plt.show()

In [ ]:
# printing a clean summary of all three datasets before moving to feature engineering
print("=== DATASET SUMMARY ===")
print(f"prices      : {prices.shape[0]:,} rows | {prices['symbol'].nunique()} tickers | {prices['date'].min().date()} to {prices['date'].max().date()}")
print(f"fundamentals: {fundamentals.shape[0]:,} rows | {fundamentals.shape[1]} columns")
print(f"securities  : {securities.shape[0]:,} rows | {securities['GICS Sector'].nunique()} sectors")

In [ ]:
import os
import json
from google.colab import drive, _message, userdata

# pulling credentials from colab secrets
github_token = userdata.get('GITHUB_TOKEN')
github_email = userdata.get('GITHUB_EMAIL')
github_name  = userdata.get('GITHUB_NAME')   # add this secret too — e.g. SidRoy97

# saving the notebook to google drive
notebook_name = '01_data_loading'  # change this per notebook — no extension
notebook_path = f'/content/drive/MyDrive/ML/stock-lens/notebooks/{notebook_name}.ipynb'

notebook_json = _message.blocking_request('get_ipynb', request='', timeout_sec=120)
with open(notebook_path, 'w') as f:
    json.dump(notebook_json['ipynb'], f)
print("saved to drive ✅")

# installing pdf conversion dependencies
os.system('apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libcups2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 -q')
os.system('pip install -q nbconvert[webpdf] playwright')
os.system('playwright install chromium')

# converting notebook to pdf
os.system(f'jupyter nbconvert --to webpdf --allow-chromium-download "{notebook_path}"')
pdf_path = notebook_path.replace('.ipynb', '.pdf')
print("pdf created!!" if os.path.exists(pdf_path) else "pdf creation failed")

# cloning repo if not already present
repo_path = '/content/stock-lens'
if not os.path.exists(repo_path):
    os.system(f'git clone https://github.com/SidRoy97/stock-lens.git {repo_path}')

# creating notebooks folder if missing
os.makedirs(f'{repo_path}/notebooks', exist_ok=True)

# copying notebook and pdf into repo
os.system(f'cp "{notebook_path}" "{repo_path}/notebooks/{notebook_name}.ipynb"')
if os.path.exists(pdf_path):
    os.system(f'cp "{pdf_path}" "{repo_path}/notebooks/{notebook_name}.pdf"')

# configuring git with user credentials
os.system(f'git -C {repo_path} config user.email "{github_email}"')
os.system(f'git -C {repo_path} config user.name "{github_name}"')
os.system(f'git -C {repo_path} add notebooks/')

commit = os.popen(f'git -C {repo_path} commit -m "update {notebook_name} notebook and pdf" 2>&1').read()
print("commit:", commit)

# pushing to github
remote = f'https://{github_token}@github.com/SidRoy97/stock-lens.git'
push   = os.popen(f'git -C {repo_path} push {remote} main 2>&1').read()
print("push:", push)